In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec

# --- Load & Preprocess ---
dataset_path = '../Email Dataset/Clean_Emails_Dataset.csv'
df = pd.read_csv(dataset_path)
df = df.dropna(subset=['Email Text', 'Email Type'])
df['label_num'] = df['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})

X = df['Email Text'].astype(str)
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Folder to save models ---
os.makedirs('../Models', exist_ok=True)

# --- TF-IDF Vectorizer (shared by LR and NB) ---
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

with open('../Models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print('Saved: tfidf_vectorizer.pkl')

# --- Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

with open('../Models/lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
print('Saved: lr_model.pkl')

# --- Naive Bayes ---
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

with open('../Models/nb_model.pkl', 'wb') as f:
    pickle.dump(nb_model, f)
print('Saved: nb_model.pkl')

# --- Word2Vec + Random Forest ---
X_train_tokens = [text.split() for text in X_train]

w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)
w2v_model.save('../Models/w2v_model.model')
print('Saved: w2v_model.model')

def get_avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_train_w2v = np.array([get_avg_vector(tokens, w2v_model) for tokens in X_train_tokens])

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_w2v, y_train)

with open('../Models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print('Saved: rf_model.pkl')

print('\nTraditional models trained and saved to ../Models/')

In [ ]:
import transformers
from packaging import version

print('transformers:', transformers.__version__)

if version.parse(transformers.__version__) >= version.parse('5.0.0'):
    raise RuntimeError(
        "This notebook expects TensorFlow-compatible Hugging Face APIs from transformers 4.x. "
        "Please run: %pip install 'transformers==4.46.3' tf-keras"
    )

print('Version check passed: transformers 4.x detected.')

In [ ]:
import os
import numpy as np
import tensorflow as tf
from transformers import DistilBertTokenizerFast, TFAutoModelForSequenceClassification

# --- DistilBERT Fine-tuning with TensorFlow (run after Cell 0) ---
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors='np'
)

train_labels = y_train.reset_index(drop=True).to_numpy(dtype=np.int32)

train_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': train_encodings['input_ids'],
        'attention_mask': train_encodings['attention_mask']
    },
    train_labels
)).shuffle(1000).batch(8)

distilbert_model = TFAutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy('accuracy')]

distilbert_model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
distilbert_model.fit(train_dataset, epochs=2)

distilbert_save_dir = '../Models/distilbert_model'
os.makedirs(distilbert_save_dir, exist_ok=True)
distilbert_model.save_pretrained(distilbert_save_dir)
tokenizer.save_pretrained(distilbert_save_dir)
print('Saved: distilbert_model/')

In [ ]:
import numpy as np
import pickle
import tensorflow as tf
from gensim.models import Word2Vec
from transformers import DistilBertTokenizerFast, TFAutoModelForSequenceClassification

# --- Load all models ---
with open('../Models/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)

with open('../Models/lr_model.pkl', 'rb') as f:
    lr_model = pickle.load(f)

with open('../Models/nb_model.pkl', 'rb') as f:
    nb_model = pickle.load(f)

with open('../Models/rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

w2v_model = Word2Vec.load('../Models/w2v_model.model')

distilbert_dir = '../Models/distilbert_model'
distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained(distilbert_dir)
distilbert_model = TFAutoModelForSequenceClassification.from_pretrained(distilbert_dir)

# --- Helper ---
def get_avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

def label(pred):
    return '🚨 Phishing' if pred == 1 else '✅ Safe'

# --- Main prediction function ---
def predict_email(email_text):
    print('\n' + '=' * 50)
    print('📧 Analyzing Email...')
    print('=' * 50)

    # TF-IDF transform
    tfidf_vec = tfidf_vectorizer.transform([email_text])

    # Word2Vec transform
    tokens = email_text.split()
    w2v_vec = np.array([get_avg_vector(tokens, w2v_model)])

    # DistilBERT transform
    bert_inputs = distilbert_tokenizer(
        [email_text],
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors='tf'
    )

    # --- Logistic Regression ---
    lr_pred = lr_model.predict(tfidf_vec)[0]
    lr_conf = lr_model.predict_proba(tfidf_vec)[0][lr_pred] * 100
    print(f'\n[Logistic Regression]  {label(lr_pred)}  ({lr_conf:.1f}% confident)')

    # --- Naive Bayes ---
    nb_pred = nb_model.predict(tfidf_vec)[0]
    nb_conf = nb_model.predict_proba(tfidf_vec)[0][nb_pred] * 100
    print(f'[Naive Bayes]          {label(nb_pred)}  ({nb_conf:.1f}% confident)')

    # --- Random Forest ---
    rf_pred = rf_model.predict(w2v_vec)[0]
    rf_conf = rf_model.predict_proba(w2v_vec)[0][rf_pred] * 100
    print(f'[Random Forest]        {label(rf_pred)}  ({rf_conf:.1f}% confident)')

    # --- DistilBERT ---
    outputs = distilbert_model(bert_inputs)
    probs = tf.nn.softmax(outputs.logits, axis=1).numpy()[0]
    db_pred = int(np.argmax(probs))
    db_conf = float(probs[db_pred] * 100)
    print(f'[DistilBERT]           {label(db_pred)}  ({db_conf:.1f}% confident)')

    # --- Majority Vote ---
    votes = [lr_pred, nb_pred, rf_pred, db_pred]
    final = 1 if sum(votes) >= 2 else 0
    print(f'\n>>> Majority Verdict:  {label(final)}')
    print('=' * 50)


# --- Try it out ---
email = """
Dear user, your account has been suspended.
Click here immediately to verify your identity and avoid permanent ban: http://secure-login-verify.com
"""

predict_email(email)

Ignore this:

In [ ]:
%pip install "transformers==4.46.3" tf-keras

In [ ]:
%pip install sklearn